In [0]:
# -------------------------
# CREATE TARGET TABLE IF NOT EXISTS
# -------------------------
spark.sql("""
CREATE TABLE IF NOT EXISTS CUSTOMER_HEALTH.GOLD.LEADS_CURRENT_STATUS
(
    LEAD_ID STRING,
    DATE_UPDATED TIMESTAMP,
    CSM STRING,
    CSM_EMAIL STRING,
    TIER STRING,
    CHANNEL_ID STRING,
    CONTRACT_START_DATE STRING,
    CONTRACT_END_DATE STRING,
    FUNNEL STRING,
    STATUS_LABEL STRING,
    DESCRIPTION STRING,
    ADD_ONS STRING,
    CREDIT_SCORE STRING,
    PHONE STRING,
    INSERT_DATE TIMESTAMP,
    UPDATED_AT TIMESTAMP
)
USING DELTA
""")

print("✅ Target table created/verified")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# -------------------------
# INCREMENTAL LOAD - Check max insert_date in target
# -------------------------
max_insert_date = spark.sql("""
    SELECT COALESCE(
        MAX(INSERT_DATE),
        TIMESTAMP('1900-01-01')
    ) AS max_dt
    FROM CUSTOMER_HEALTH.GOLD.LEADS_CURRENT_STATUS
""").first()["max_dt"]

print(f"Max insert_date in target: {max_insert_date}")

# -------------------------
# TABLES - MATCH SNOWFLAKE LOGIC
# -------------------------
leads_raw_all = spark.table("CUSTOMER_HEALTH.BRONZE.LEADS_RAW")

# Filter for new records only
leads_raw = (
    leads_raw_all
    .filter(F.col("insert_date") > F.lit(max_insert_date))
)

# Check if there are new records
if leads_raw.limit(1).count() == 0:
    print("No new records found - exiting")
    dbutils.notebook.exit("No new records found")

print(f"Processing {leads_raw.count()} new records")

lead_merges_raw = spark.table("CUSTOMER_HEALTH.BRONZE.LEAD_MERGES")
users = spark.table("CUSTOMER_HEALTH.SILVER.CLOSE_CRM_USERS_PROCESSED")

# -------------------------
# PARSE LEAD MERGES JSON
# -------------------------
lead_merges = (
    lead_merges_raw
    .withColumn(
        "SOURCE_LEAD_ID",
        F.get_json_object("raw_data", "$.SOURCE_LEAD_ID")
    )
    .withColumn(
        "DESTINATION_LEAD_ID",
        F.get_json_object("raw_data", "$.DESTINATION_LEAD_ID")
    )
    .select("SOURCE_LEAD_ID", "DESTINATION_LEAD_ID")
)

# -------------------------
# USERS DIM
# -------------------------
users_df = (
    users.select(
        "user_id",
        F.col("email").alias("csm_email"),
        F.concat_ws(" ", "first_name", "last_name").alias("csm")
    )
)

# -------------------------
# FLATTENED - MATCH SNOWFLAKE: FLATTEN contacts AND extract lead custom fields
# -------------------------
# Parse JSON
parsed_df = (
    leads_raw
    .withColumn(
        "json_object",
        F.expr("customer_health.silver.parse_activity_final(raw_data)")
    )
)

# Flatten contacts array
flattened = (
    parsed_df
    .withColumn(
        "contact",
        F.explode(
            F.from_json(
                F.get_json_object("json_object", "$.contacts"),
                """
                array<
                    struct<
                        lead_id:string,
                        date_updated:string,
                        phones:array<struct<phone:string>>
                    >
                >
                """
            )
        )
    )
)

# Extract contact info AND lead custom fields from json_object
contacts_df = (
    flattened
    .select(
        "contact.lead_id",
        "json_object",
        F.to_timestamp("contact.date_updated").alias("date_updated"),
        F.expr("get(contact.phones, 0).phone").alias("phone"),
        "insert_date"
    )
)

# -------------------------
# ENRICHMENT - MATCH SNOWFLAKE: Extract lead custom fields AND join
# -------------------------
enriched = (
    contacts_df
    .join(
        lead_merges,
        contacts_df["lead_id"] == lead_merges["SOURCE_LEAD_ID"],
        "left"
    )
    .join(
        users_df,
        users_df["user_id"] == F.get_json_object(F.col("json_object"), "$.custom.CSM"),
        "left"
    )
    .withColumn(
        "lead_id",
        F.coalesce(
            F.col("DESTINATION_LEAD_ID"),
            F.col("lead_id")
        )
    )
    .withColumn("raw_lead_id", F.col("lead_id"))
    .withColumn("tier", F.get_json_object("json_object", "$.custom.Tier"))
    .withColumn("contract_start_date", F.get_json_object("json_object", '$.custom."Contract Start Date"'))
    .withColumn("contract_end_date", F.get_json_object("json_object", '$.custom."Contract End Date"'))
    .withColumn("funnel", F.get_json_object("json_object", "$.custom.Funnel"))
    .withColumn("channel_id", F.get_json_object("json_object", "$.custom.Channel_ID"))
    .withColumn("status_label", F.get_json_object("json_object", "$.status_label"))
    .withColumn("description", F.get_json_object("json_object", "$.description"))
    .withColumn("add_ons", F.get_json_object("json_object", '$.custom."Add-ons"'))
    .withColumn("credit_score", F.get_json_object("json_object", '$.custom."[PF] Estimated Credit Score"'))
    .select(
        "lead_id",
        "raw_lead_id",
        "date_updated",
        "csm",
        "csm_email",
        "tier",
        "contract_start_date",
        "contract_end_date",
        "funnel",
        "channel_id",
        "status_label",
        "description",
        "add_ons",
        "credit_score",
        "phone",
        "insert_date"
    )
)

# -------------------------
# FILTER - MATCH SNOWFLAKE: WHERE tier IS NOT NULL
# -------------------------
enriched = enriched.filter(F.col("tier").isNotNull())

# -------------------------
# DEDUP - MATCH SNOWFLAKE: QUALIFY ROW_NUMBER()
# -------------------------
window_spec = Window.partitionBy("lead_id").orderBy(
    F.col("date_updated").desc(),
    F.col("insert_date").desc()
)

deduped = (
    enriched
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# -------------------------
# TARGET LEAD MERGE FIX STEP (UPDATE MERGED LEAD IDS FIRST)
# -------------------------
# Update any existing records that have SOURCE_LEAD_ID to use DESTINATION_LEAD_ID
# This prevents duplicate records after lead merges

gold_table = DeltaTable.forName(
    spark, "CUSTOMER_HEALTH.GOLD.LEADS_CURRENT_STATUS"
)

# Update existing records with merged lead IDs
gold_table.alias("t").merge(
    lead_merges.alias("lm"),
    "t.LEAD_ID = lm.SOURCE_LEAD_ID"
).whenMatchedUpdate(
    set={
        "LEAD_ID": "lm.DESTINATION_LEAD_ID",
        "UPDATED_AT": "current_timestamp()"
    }
).execute()

# -------------------------
# DELTA MERGE (MERGE NEW DATA)
# -------------------------

(
    gold_table.alias("t").merge(
        deduped.alias("s"),
        "t.LEAD_ID = s.lead_id"
    )
    .whenMatchedUpdate(
        condition="""
            t.CSM <> s.csm OR
            t.TIER <> s.tier OR
            t.STATUS_LABEL <> s.status_label OR
            t.ADD_ONS <> s.add_ons OR
            t.CREDIT_SCORE <> s.credit_score OR
            t.PHONE <> s.phone
        """,
        set={
            "DATE_UPDATED": "s.date_updated",
            "CSM": "s.csm",
            "CSM_EMAIL": "s.csm_email",
            "TIER": "s.tier",
            "CHANNEL_ID": "s.channel_id",
            "CONTRACT_START_DATE": "s.contract_start_date",
            "CONTRACT_END_DATE": "s.contract_end_date",
            "FUNNEL": "s.funnel",
            "STATUS_LABEL": "s.status_label",
            "DESCRIPTION": "s.description",
            "ADD_ONS": "s.add_ons",
            "CREDIT_SCORE": "s.credit_score",
            "PHONE": "s.phone",
            "INSERT_DATE": "s.insert_date",
            "UPDATED_AT": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "LEAD_ID": "s.lead_id",
            "DATE_UPDATED": "s.date_updated",
            "CSM": "s.csm",
            "CSM_EMAIL": "s.csm_email",
            "TIER": "s.tier",
            "CHANNEL_ID": "s.channel_id",
            "CONTRACT_START_DATE": "s.contract_start_date",
            "CONTRACT_END_DATE": "s.contract_end_date",
            "FUNNEL": "s.funnel",
            "STATUS_LABEL": "s.status_label",
            "DESCRIPTION": "s.description",
            "ADD_ONS": "s.add_ons",
            "CREDIT_SCORE": "s.credit_score",
            "PHONE": "s.phone",
            "INSERT_DATE": "s.insert_date",
            "UPDATED_AT": "current_timestamp()"
        }
    )
    .execute()
)

print(f"\n✅ Pipeline executed successfully")
print(f"Processed {deduped.count()} unique leads (after dedup and tier filter)")